In [1]:
import json
import pandas as pd

df = pd.read_parquet("../data/processed/clean_products.parquet")
report = json.load(open("../data/processed/data_quality.json"))

print("Rows:", len(df))
print("Duplicate IDs:", df["product_id"].duplicated().sum())
print("Selling price above MRP:", (df["price"] > df["mrp"]).sum())
print("Price or MRP <= 0:", ((df["price"] <= 0) | (df["mrp"] <= 0)).sum())
print("Rating equal to 0:", (df["rating"] == 0).sum())
print("Has rating but rating missing:", (df["has_rating"] & df["rating"].isna()).sum())
print("No rating flag but has a rating:", (~df["has_rating"] & df["rating"].notna()).sum())
print("Highest discount_pct:", round(df["discount_pct"].max(), 1))
print("Report rows kept equals file rows:", report["steps"][-1]["remaining"] == len(df))
print("Segments below 30 listings:", report["segments_below_minimum"])
print(df["segment"].value_counts())

Rows: 18037
Duplicate IDs: 0
Selling price above MRP: 0
Price or MRP <= 0: 0
Rating equal to 0: 0
Has rating but rating missing: 0
No rating flag but has a rating: 0
Highest discount_pct: 90.0
Report rows kept equals file rows: True
Segments below 30 listings: []
segment
Women's Kurtas      6212
Men's Kurtas        5851
Women's Sneakers    3308
Women's Shirts      1628
Men's Shirts        1038
Name: count, dtype: int64


In [2]:
diff = (df["discount_pct"] - df["file_discount"]).abs()
close = diff <= 1
rupees = df["file_discount"] == (df["mrp"] - df["price"])

print("Within 1 point of the file's discount:", close.sum())
print("File's discount equals rupees off:", rupees.sum())
print("Neither:", (~close & ~rupees).sum())
df.loc[~close, ["name", "price", "mrp", "discount_pct", "file_discount"]].sample(8, random_state=42)

Within 1 point of the file's discount: 16717
File's discount equals rupees off: 4044
Neither: 0


,name,price,mrp,discount_pct,file_discount
9197,Women Printed Kurta,2690.0,6500.0,58.615385,3810
9521,Floral Printed Plus Size Shirt,749.0,2499.0,70.028011,1750
8059,Women Geometric Embroidered Georgette Kurta,1499.0,2999.0,50.016672,1500
1270,Casual Stripes Printed Shirt,899.0,1699.0,47.086521,800
17502,Men Solid Kurta,1699.0,2799.0,39.299750,1100
5849,Casual Shirt,899.0,2499.0,64.025610,1600
6972,Camouflage Printed Shirt,1299.0,2495.0,47.935872,1196
11961,Notched Neck Embroidered Kurta,749.0,1999.0,62.531266,1250


In [3]:
pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 220)


def look(segment, n=50, seed=1):
    rows = df[df["segment"] == segment].sample(n, random_state=seed).copy()
    rows["link_name"] = rows["url"].str.split("/").str[5]
    return rows[["name", "link_name"]].reset_index(drop=True)


look("Men's Shirts")


,name,link_name
0,Men Plus Size Casual Shirt,sojanya-plus-men-grey-classic-casual-shirt
1,Plus Size Pure Cotton Shirt,sztori-men-plus-size-blue-pure-cotton-solid-casual-shirt
2,Plus Size Men Slim Fit Casual Shirt,john-pride-plus-size-men-yellow-slim-fit-solid-casual-shirt
3,Men Solid Ethnic Shirt,vastramay-men-black-ethnic-shirt
4,Solid Slim Fit Casual Shirt,mod-ecru-men-blue-solid-straight-slim-fit-casual-shirt
5,Men Casual Cotton Shirt,pluss-men-printed-casual-cotton-shirt
6,Plus Size Cotton Casual Shirt,instafab-plus-size-men-classic-multi-striped-cotton-casual-shirt
7,Casual Shirt,plus-size-men-blue-casual-shirt
8,Casual Shirt,bigbanana-plus-size-men-red-comfort-gingham-checks-checked-casual-shirt
9,Plus Size Solid Casual Shirt,vastramay-plus-men-plus-size-black-premium-solid-casual-shirt


In [4]:
look("Women's Shirts")

,name,link_name
0,Casual Shirt,patrorna-women-blue-comfort-casual-shirt
1,Casual Shirt,crimsoune-club-women-blue-casual-shirt
2,Embroidered Casual Shirt,house-of-kkarma-women-cream-coloured--pink-floral-embroidered-opaque-casual-shirt
3,Classic Fit Crop Shirt,brinns-women-classic-fit-spread-collar-crop-shirt
4,Solid Slim Fit Formal Shirt,van-heusen-woman-women-solid-slim-fit-formal-shirt
5,Women Casual Cotton Shirt,showoff-women-tartan-checks-casual-cotton-shirt
6,Casual Shirt,divena-women-blue-comfort-floral-printed-casual-shirt
7,Relaxed Casual Shirt,atun-women-relaxed-casual-shirt
8,Women Striped Casual Shirt,her-by-invictus-women-yellow--blue-striped-casual-shirt
9,Women Cotton Casual Shirt,forever-21-women-cotton-casual-shirt


In [5]:
look("Men's Kurtas")

,name,link_name
0,Men Solid Kurta,riag-men-blue-solid-kurta
1,Men Mandarin Collar Kurta,anug-by-sojanya-men-black-mandarin-collar-kurta
2,Men Long Sleeves Mandarin Collar Kurta,herenow-men-thread-work-long-sleeves-mandarin-collar-kurta
3,Solid Straight Kurta,manyavar-men-green-solid-straight-kurta
4,Men Woven Design Kurta,sojanya-men-blue--black-woven-design-kurta
5,Solid Straight Kurta,sojanya-men-navy-blue-solid-straight-kurta
6,Men Kurta,jompers-men-mustard-yellow-kurta
7,Men A-Line Straight Kurta,thangamagan-men-teal-blue-solid-band-collar-a-line-straight-kurta
8,Men Striped Kurta,kisah-men-maroon-striped-kurta
9,Men Printed Kurta,deyann-men-navy-blue-ethnic-motifs-printed-kurta


In [6]:
look("Women's Kurtas")

,name,link_name
0,Women Printed Kurta,prakrti-women-ethnic-motifs-printed-tie-up-neck-kurta
1,Women Kurta With Dupatta,deebaco-women-red-kurta-with-dupatta
2,Women Chikankari Kurta,anouk-women-ethnic-motifs-chikankari-embroidered-kurta
3,Women Yoke Design Kurta,rb-women-peach-coloured-yoke-design-keyhole-neck-kurta
4,Women Geometric Printed Kurta,rangriti-women-geometric-printed-kurta
5,Women Printed Cotton Kurta,soundarya-women-abstract-printed-cotton-kurta
6,Women Ethnic Motifs Printed Kurta,anouk-women-brown-ethnic-motifs-printed-kurta
7,Women Paisley Printed Anarkali Kurta,w-women-paisley-printed-anarkali-kurta
8,V-Neck Pure Cotton Kurta,fabindia-women-v-neck-pure-cotton-kurta
9,Women Quirky Printed Kurta,mauka-women-quirky-printed-kurta


In [7]:
look("Women's Sneakers")


,name,link_name
0,Women Printed Slip-On Sneakers,kazarmax-women-printed-slip-on-sneakers
1,Women Comfort Insole Sneakers,call-it-spring-women-gigii-comfort-insole-basics-sneakers
2,Women Lace-Ups Sneakers,bonkerz-women-lace-ups-sneakers
3,Women Colourblocked Sneakers,shezone-women-white--blue-colourblocked-sneakers
4,Women Textured Mesh Sneakers,arivo-women-textured-mesh-comfort-insole-sneakers
5,Women Lightweight Sneakers,el-paso-women-colourblockedv-lightweight--lace-up-sneakers
6,Women Solid Tanjun Sneakers,nike-women-solid-tanjun-regular-sneakers
7,Women Woven Design Slip-On Sneakers,carlton-london-sports-women-woven-design-slip-on-sneakers
8,Women Mile Rider Sneakers,puma-women-mile-rider-sunny-gataway-sneakers
9,Women Striped Sneakers,kraasa-women-round-toe-lace-ups-striped-sneakers
